# **(1)** Chemprop for peptides properties predictions
### Single-Task vs Multi-Task Regression

| | Single-Task | Multi-Task |
|---|---|---|
| **Model** | 3 MPNN, 1 for each y | 1 shared MPNN, 3 outputs |
| **`y` for datapoint** | `shape (1,)` | `shape (3,)` |
| **`RegressionFFN`** | `n_tasks=1` × 3 | `n_tasks=3` × 1 |
| **Shared Backbone** | ❌ | ✅ |
| **NaN** | Filtered out for the missing ones | loss masked |

The aim of this tutorial is familiarise with different chemprop configuration to predict three different properties (regression task) having as input the SMILES strings of peptides and as ys the following targets:

### Targets
| # | CSV column name | Description |
|---|-------------|-------------|
| 1 | `purity` | HPLC purity of crude samples (0–100%) |
| 2 | `camsol_score` | CamSol score (proxy of solubility) |
| 3 | `revised_50hemo` | peptide concentration of 50% hemolysis (µM) |

Additional resources and chemprop tutorials can be found here: https://chemprop.readthedocs.io/en/latest/notebooks.html

### Dependences to run it locally
```bash
pip install chemprop lightning rdkit pandas
```

### To run the notebook change runtime type to GPU!


In [ ]:
pip install chemprop lightning rdkit pandas

In [ ]:
import os

if os.getenv("COLAB_RELEASE_TAG"):
    !git clone https://github.com/rbirolo/IX_technical_turorial2026.git
    %cd IX_technical_turorial2026

In [ ]:
import warnings, pickle
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import lightning as pl
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

from chemprop.data import (
    MoleculeDatapoint, MoleculeDataset, build_dataloader,
    make_split_indices, split_data_by_indices,
)
from chemprop.models import MPNN
from chemprop.nn import BondMessagePassing, MeanAggregation, RegressionFFN, RMSE, MAE


## Configuration

In [ ]:
# Config
data_file = "data/experimental_dataset.csv"          #input file
ST_output = Path("chemprop_st")  # output single-task
MT_output = Path("chemprop_mt")   # output multi-task
targets = ["purity", "camsol_score", "revised_50hemo"]

In [ ]:
# Architecture hyperparameters
MP_DEPTH   = 3
MP_DIM     = 300
FFN_LAYERS = 2
FFN_DIM    = 300
DROPOUT    = 0.2

# Training
MAX_EPOCHS = 50
INIT_LR    = 1e-4
MAX_LR     = 1e-3
FINAL_LR   = 1e-4
WARMUP_EPS = 2
BATCH_SIZE = 64
SEED       = 42

# Split sizes for train, validation, and test sets
SPLIT_SIZES = (0.70, 0.15, 0.15)

## Data load and cleaning

CSV file with: `SMILES`, `purity`, `camsol_score`, `revised_50hemo` columns.


In [ ]:
def load_dataset(data_file: str):

    df = pd.read_csv(data_file)

    for t in targets:
        df[t] = pd.to_numeric(df[t], errors="coerce")
    df = df.dropna(subset=["SMILES"]).copy()

    mols, valid_idx = [], []
    for i, smi in enumerate(df["SMILES"]):
        mol = Chem.MolFromSmiles(str(smi))
        if mol is not None:
            mols.append(mol); valid_idx.append(i)
        else:
            print(f"Invalid SMILES (row_{i}): {str(smi)[:100]}…")

    df  = df.iloc[valid_idx].reset_index(drop=True)
    ys  = df[targets].values.astype(float)
    return mols, ys, df

mols, ys, df_clean = load_dataset(data_file)
N = len(mols)

print(f"{'Target':<22} {'entries':>8} {'mean':>9} {'std':>7} {'min':>8} {'max':>8}")
print("─" * 66)
for i, t in enumerate(targets):
    v = ys[:, i][~np.isnan(ys[:, i])]
    print(f"{t:<22} {len(v):>8} {v.mean():>9.4f} {v.std():>7.4f} {v.min():>8.4f} {v.max():>8.4f}")


## Dataset visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ["#4C72B0", "#55A868", "#C44E52"]
for ax, t, c in zip(axes, targets, colors):
    v = ys[:, targets.index(t)]; v = v[~np.isnan(v)]
    ax.hist(v, bins=30, color=c, edgecolor="white", linewidth=0.6)
    ax.axvline(v.mean(), color="black", ls="--", lw=1.2, label=f"μ={v.mean():.2f}")
    ax.set_title(t, fontsize=12, fontweight="normal");
    ax.set_xlabel("y_value"); ax.set_ylabel("number of samples")
    ax.legend(fontsize=9); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.show()


## Data split

Same split used for single- and multi-task for comparison.


In [ ]:
pl.seed_everything(SEED)

train_idxs, val_idxs, test_idxs = make_split_indices(
    mols, split="random", sizes=SPLIT_SIZES, seed=SEED
)

def flatten_split(data, idxs):
    split, _ = split_data_by_indices(data, *idxs) if False else (None, None)
    return None

_train_idx = train_idxs[0]; _val_idx = val_idxs[0]; _test_idx = test_idxs[0]

print(f"Train : {len(_train_idx):>4}  ({len(_train_idx)/N*100:.1f}%)")
print(f"Val   : {len(_val_idx):>4}  ({len(_val_idx)/N*100:.1f}%)")
print(f"Test  : {len(_test_idx):>4}  ({len(_test_idx)/N*100:.1f}%)")


## Functions for data loading, model architecture and evalutation

In [ ]:
def make_loaders(datapoints):
    train, val, test = split_data_by_indices(datapoints, train_idxs, val_idxs, test_idxs)
    train = [dp for sub in train for dp in sub]
    val = [dp for sub in val for dp in sub]
    test = [dp for sub in test for dp in sub]

    train_ds = MoleculeDataset(train)    #step0: molecular graph from SMILES
    val_ds = MoleculeDataset(val)
    test_ds = MoleculeDataset(test)

    scaler = train_ds.normalize_targets()
    val_ds.normalize_targets(scaler)

    train_load = build_dataloader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_load = build_dataloader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_load = build_dataloader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return train_load, val_load, test_load, scaler


def make_mpnn(n_tasks: int):
    mp  = BondMessagePassing(depth=MP_DEPTH, d_h=MP_DIM, dropout=DROPOUT)   #step1: message passing
    agg = MeanAggregation()                                                 #step2: embedding aggregation, pools atom embeddings to molecule embedding
    ffn = RegressionFFN(n_tasks=n_tasks, input_dim=mp.output_dim,           #step3: FFN for regression 1-head (single-task) 3-heads (multi-task)
                        hidden_dim=FFN_DIM, n_layers=FFN_LAYERS, dropout=DROPOUT)

    return MPNN(mp, agg, ffn, metrics=[RMSE(), MAE()],
                init_lr=INIT_LR, max_lr=MAX_LR, final_lr=FINAL_LR,
                warmup_epochs=WARMUP_EPS)


def evaluate_st(model, test_loader, scaler, target_name: str) -> dict:
    model.eval()
    all_p, all_gt = [], []
    with torch.no_grad():
        for batch in test_loader:
            bmg, V_d, X_d, targets, *_ = batch
            all_p.append(model(bmg, V_d, X_d).numpy())
            all_gt.append(targets.numpy())

    preds_norm = np.concatenate(all_p)          # (N, 1)
    gt_orig    = np.concatenate(all_gt)         # (N, 1)
    preds_orig = scaler.inverse_transform(preds_norm)

    gt = gt_orig[:, 0]; pr = preds_orig[:, 0]
    mask = ~np.isnan(gt)
    gt, pr = gt[mask], pr[mask]
    return {target_name: {
        "gt": gt, "pr": pr,
        "rmse": float(np.sqrt(np.mean((pr - gt)**2))),
        "mae":  float(np.mean(np.abs(pr  - gt))),
        "pearson_r": float(np.corrcoef(gt, pr)[0, 1]),
        "spearman_r": float(pd.Series(gt).corr(pd.Series(pr), method="spearman")),
        "n":    int(mask.sum()),
    }}



def evaluate_mt(model, test_loader, scaler, target_names: list) -> dict:
    model.eval()
    all_p, all_gt = [], []
    with torch.no_grad():
        for batch in test_loader:
            bmg, V_d, X_d, targets, *_ = batch
            all_p.append(model(bmg, V_d, X_d).numpy())
            all_gt.append(targets.numpy())

    preds_norm = np.concatenate(all_p)
    gt_orig    = np.concatenate(all_gt)
    preds_orig = scaler.inverse_transform(preds_norm)

    out = {}
    for i, t in enumerate(target_names):
        mask = ~np.isnan(gt_orig[:, i])
        if not mask.any():
            continue
        gt, pr = gt_orig[mask, i], preds_orig[mask, i]
        out[t] = {
            "gt": gt, "pr": pr,
            "rmse": float(np.sqrt(np.mean((pr - gt)**2))),
            "mae":  float(np.mean(np.abs(pr  - gt))),
            "pearson_r": float(np.corrcoef(gt, pr)[0, 1]),
            "spearman_r": float(pd.Series(gt).corr(pd.Series(pr), method="spearman")),
            "n":    int(mask.sum()),
        }
    return out


def run_trainer(model, train_load, val_load):
    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS, logger=False,
        enable_checkpointing=False, enable_progress_bar=True, accelerator="auto",
    )
    trainer.fit(model, train_load, val_load)
    return trainer

## A) Single-Task

One single model for each task.  

```
purity        → MPNN_1  (RegressionFFN n_tasks=1)
camsol_score  → MPNN_2  (RegressionFFN n_tasks=1)
revised_50hemo→ MPNN_3  (RegressionFFN n_tasks=1)
```


### A1 · datapoints (y shape: `(1,)` for each target, NaN removed) and splitting

In [ ]:
st_datapoints = {}

for t in targets:
    col     = ys[:, targets.index(t)]
    valid   = ~np.isnan(col)
    t_mols  = [m for m, v in zip(mols, valid) if v]
    t_ys    = col[valid].reshape(-1, 1)
    t_dps   = [MoleculeDatapoint(mol=m, y=y.flatten()) for m, y in zip(t_mols, t_ys)]
    st_datapoints[t] = (t_mols, t_dps)
    print(f"{t:<22} number of datapoints: {len(t_dps)}")


### A2 · Training (3 single models)

In [ ]:
st_models, st_scalers, st_loaders = {}, {}, {}

for t in targets:
    print(f"Training single-task: {t}")
    t_mols, t_dps = st_datapoints[t]

    # Split on the target-specific dataset (NaN reduce the number of samples)
    t_train_i, t_val_i, t_test_i = make_split_indices(
        t_mols, split="random", sizes=SPLIT_SIZES, seed=SEED
    )
    t_train, t_val, t_test = split_data_by_indices(t_dps, t_train_i, t_val_i, t_test_i)
    t_train = [dp for sub in t_train for dp in sub]
    t_val = [dp for sub in t_val for dp in sub]
    t_test = [dp for sub in t_test for dp in sub]

    train_ds = MoleculeDataset(t_train); val_ds = MoleculeDataset(t_val); test_ds = MoleculeDataset(t_test)
    scaler = train_ds.normalize_targets(); val_ds.normalize_targets(scaler)
    train_load = build_dataloader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_load = build_dataloader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_load = build_dataloader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = make_mpnn(n_tasks=1)
    run_trainer(model, train_load, val_load)

    st_models[t]  = model
    st_scalers[t] = scaler
    st_loaders[t] = test_load


### A3 · Single-task evaluation

In [ ]:
st_metrics = {}
for t in targets:
    # evaluate_st: modello n_tasks=1, gt shape (N,1)
    m = evaluate_st(st_models[t], st_loaders[t], st_scalers[t], t)
    st_metrics[t] = m[t]
    print(f" {t:<22}  RMSE={st_metrics[t]['rmse']:.4f}  MAE={st_metrics[t]['mae']:.4f} pearson={st_metrics[t]['pearson_r']:.4f} spearman={st_metrics[t]['spearman_r']:.4f} N={st_metrics[t]['n']}")


### A4 · Model saving single-task

In [ ]:
ST_output.mkdir(parents=True, exist_ok=True)
for t in targets:
    torch.save(st_models[t].state_dict(), ST_output / f"{t}_model.pt")
    with open(ST_output / f"{t}_scaler.pkl", "wb") as f:
        pickle.dump(st_scalers[t], f)
print(f"Models saved to: {ST_output}/")


## B) Multi-Task

Single MPNN with shared backbone and **3 output**.
Missing NaN are masked in the loss

```
SMILES → BondMessagePassing(shared)
       → MeanAggregation
       → RegressionFFN(n_tasks=3)
       → [purity, camsol_score, revised_50hemo]
```


### B1 · Datapoints (y shape: `(3,)`, NaN kept)

In [ ]:
mt_datapoints = [
    MoleculeDatapoint(mol=m, y=y.flatten())
    for m, y in zip(mols, ys)
]
print(f"{len(mt_datapoints)} datapoints multi-task (y shape: {len(targets)},))")
print(f"NaN for target: {[int(np.isnan(ys[:,i]).sum()) for i in range(len(targets))]}")


### B2 · Training

In [ ]:
mt_train_load, mt_val_load, mt_test_load, mt_scaler = make_loaders(mt_datapoints)

mt_model = make_mpnn(n_tasks=len(targets))
n_params  = sum(p.numel() for p in mt_model.parameters())

run_trainer(mt_model, mt_train_load, mt_val_load)

### B3 · Multi-task evaluation metrics

In [ ]:
mt_metrics = evaluate_mt(mt_model, mt_test_load, mt_scaler, targets)

for t in targets:
    if t in mt_metrics:
        m = mt_metrics[t]
        print(f"  {t:<22}  RMSE={m['rmse']:.4f}  MAE={m['mae']:.4f} pearson={m['pearson_r']:.4f} spearman={m['spearman_r']:.4f} N={m['n']}")


### B4 · Multi-task model saving

In [ ]:
MT_output.mkdir(parents=True, exist_ok=True)
torch.save(mt_model.state_dict(), MT_output / "multitask_model.pt")
with open(MT_output / "multitask_scaler.pkl", "wb") as f:
    pickle.dump(mt_scaler, f)
print(f"Multi-task model saved to: {MT_output}/")


## C) Single-Task vs Multi-Task


In [ ]:
# Plotting predictions vs real labels
fig, axes = plt.subplots(1, len(targets), figsize=(5.5 * len(targets), 5))

for ax, t in zip(axes, targets):
    st = st_metrics.get(t); mt = mt_metrics.get(t)
    if not st or not mt:
        ax.set_visible(False); continue

    all_vals = np.concatenate([st["gt"], st["pr"], mt["gt"], mt["pr"]])
    lo, hi   = all_vals.min() * 0.95, all_vals.max() * 1.05

    ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="y = x", zorder=1)

    ax.scatter(st["gt"], st["pr"], color="#4C72B0", edgecolors="white", s=65,
               linewidths=0.5, alpha=0.85, label=f"Single-task RMSE={st['pearson_r']:.3f}", zorder=3)
    ax.scatter(mt["gt"], mt["pr"], color="#E87A2A", edgecolors="white", s=65,
               linewidths=0.5, alpha=0.85,
               label=f"Multi-task Pearson r={mt['pearson_r']:.3f}", zorder=2)

    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_title(t, fontsize=12, fontweight="normal")
    ax.set_xlabel("y_value"); ax.set_ylabel("predicted")
    ax.legend(fontsize=9); ax.spines[["top","right"]].set_visible(False)

plt.tight_layout(); plt.show()
